# Chấm điểm rủi ro tín dụng trên Google Colab

Notebook này chạy toàn bộ repo trên Colab **CPU** (không cần GPU):

1. Cài thư viện và lấy dữ liệu Home Credit từ Kaggle (hoặc dùng dữ liệu giả lập).
2. Huấn luyện baseline + LightGBM, tinh chỉnh bằng Optuna, chọn ngưỡng theo chi phí.
3. Xem SHAP: yếu tố nào làm tăng rủi ro.
4. Gọi thử API ngay trong notebook.
5. (Dự án #25) Giả lập 12 tháng drift và retrain tự động.
6. Lưu mô hình vào Google Drive.

> Số liệu đưa vào README/CV phải lấy từ lần chạy trên **dữ liệu thật**, không lấy từ dữ liệu giả lập.

In [ ]:
# 1. Lấy code. Thay tak2k5 bằng tài khoản của bạn sau khi đã đẩy repo lên GitHub.
REPO_URL = "https://github.com/tak2k5/risk-ml-platform.git"

import os
if not os.path.exists("risk-ml-platform"):
    !git clone -q {REPO_URL}
%cd risk-ml-platform/credit
!pip install -q -r requirements.txt matplotlib
import sys; sys.path[:0] = ["src", "../riskops/src"]  # riskops: thư viện dùng chung của repo

## 2. Dữ liệu

Notebook tự chọn:
- **Dữ liệu thật** khi Colab Secrets có `KAGGLE_USERNAME` và `KAGGLE_KEY` (lấy trong `kaggle.json`), và bạn đã bấm
  Join Competition ở trang [Home Credit Default Risk](https://www.kaggle.com/competitions/home-credit-default-risk).
- **Dữ liệu giả lập** khi chưa có hai khóa đó: chỉ để thử đường ống, báo cáo ghi vào `reports/synthetic/`, **không** lấy số.

In [ ]:
from google.colab import userdata

def secret(name):
    """Đọc Colab Secrets (biểu tượng chìa khóa bên trái); chưa có thì trả None thay vì báo lỗi."""
    try:
        return userdata.get(name) or None
    except Exception:
        return None

if secret("KAGGLE_USERNAME") and secret("KAGGLE_KEY"):
    os.environ.update(KAGGLE_USERNAME=secret("KAGGLE_USERNAME"), KAGGLE_KEY=secret("KAGGLE_KEY"))
    !bash scripts/download_data.sh
    USE_SYNTHETIC = False
    print("DỮ LIỆU THẬT: Home Credit")
else:
    USE_SYNTHETIC = True
    os.environ["REPORTS_DIR"] = "reports/synthetic"   # số giả lập không lẫn vào báo cáo thật
    print("CHÚ Ý: chưa có KAGGLE_USERNAME / KAGGLE_KEY trong Colab Secrets -> ĐANG DÙNG DỮ LIỆU GIẢ LẬP, không lấy số này.")

## 3. Huấn luyện

- `--tune 30`: 30 lần thử Optuna (khoảng vài phút trên CPU Colab với dữ liệu thật).
- Mô hình, `metadata.json` và mẫu tham chiếu được lưu trong `models/`, bảng kết quả trong `reports/metrics.md`.

In [ ]:
if USE_SYNTHETIC:
    !PYTHONPATH=src:../riskops/src python -m credit_risk.train --synthetic 20000 --tune 10
else:
    !PYTHONPATH=src:../riskops/src python -m credit_risk.train --tune 30

In [ ]:
from IPython.display import Markdown, display
display(Markdown(open(os.path.join(os.environ.get("REPORTS_DIR", "reports"), "metrics.md")).read()))

## 4. Giải thích mô hình bằng SHAP

- **Độ quan trọng tổng thể**: trung bình |SHAP| trên 1000 hồ sơ.
- **Một hồ sơ cụ thể**: 3 yếu tố đẩy rủi ro lên nhiều nhất (chính là trường `reasons` mà API trả về).

In [ ]:
import json, joblib
import matplotlib.pyplot as plt
from credit_risk.config import RAW_COLS, get_settings
from credit_risk.data import load_raw, make_synthetic
from credit_risk.explain import Explainer, global_importance, readable

s = get_settings()
pipe = joblib.load(s.model_path)
df = make_synthetic(5000, seed=1) if USE_SYNTHETIC else load_raw(s.data_path, nrows=20000)
X = df[RAW_COLS]

explainer = Explainer(pipe)
imp = global_importance(explainer, X.sample(1000, random_state=0)).head(12)
imp.index = [readable(f) for f in imp.index]
imp[::-1].plot.barh(figsize=(7, 5), title="Độ quan trọng (mean |SHAP|)")
plt.tight_layout(); plt.show()

for r in explainer.top_reasons(X.head(1))[0]:
    print(f"{r.label:45s} +{r.contribution:.3f}")

## 5. Gọi thử API ngay trong notebook

`TestClient` chạy FastAPI trong cùng tiến trình, không cần mở cổng mạng.

In [ ]:
from fastapi.testclient import TestClient
from credit_risk.api import app
from credit_risk.schemas import Application

example = Application.model_config["json_schema_extra"]["example"]
with TestClient(app) as client:
    r = client.post("/predict", json=example)
    print(r.status_code, r.headers.get("X-Process-Time-Ms"), "ms")
    print(json.dumps(r.json(), indent=2, ensure_ascii=False))
    bad = client.post("/predict", json={**example, "AMT_INCOME_TOTAL": -5})
    print("Dữ liệu sai ->", bad.status_code)

## 6. (Dự án #25) Giám sát drift và retrain tự động

Giả lập 12 tháng: khách dần trẻ hơn, thu nhập thấp hơn, và một cú sốc kinh tế ở tháng 6.
So sánh mô hình **để nguyên** với mô hình **tự retrain khi phát hiện drift**.

In [ ]:
!PYTHONPATH=src:../riskops/src python -m pipelines.simulate --months 12 2>&1 | tail -15
from IPython.display import Image
display(Markdown(open("reports/sim/simulation.md").read()))
display(Image("reports/sim/simulation.png"))

## 7. Lưu mô hình vào Google Drive

Colab xóa mọi thứ khi hết phiên, nên chép `models/` và `reports/` sang Drive.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
!mkdir -p /content/drive/MyDrive/credit-risk && cp -r models reports /content/drive/MyDrive/credit-risk/
!ls /content/drive/MyDrive/credit-risk/models

## Đẩy kết quả lên GitHub

Đẩy các file trong `credit/reports/` lên repo của bạn, không cần tải về. Cần `GH_TOKEN` trong Colab Secrets
(xem hướng dẫn, Bước 0). Chưa có token thì ô này chỉ nhắc, kết quả vẫn nằm trên Drive.
Chạy ở chế độ giả lập thì ô này không đẩy gì.

In [ ]:
if USE_SYNTHETIC:
    print("Đang ở chế độ GIẢ LẬP: không đẩy kết quả lên GitHub.")
else:
    token = secret("GH_TOKEN")
    if token:
        os.environ["GH_TOKEN"] = token
    !python ../scripts/colab_push.py credit/reports "Tín dụng: kết quả trên Home Credit (Colab)"